# Repaso pandas


### Descripción de los conjuntos de datos

Este ejercicio se basa en el análisis de álbumes de música rock, bandas y valoraciones de usuarios. Trabajarás con los siguientes tres ficheros:

---

#### `albums_rock.csv`
Contiene información detallada sobre álbumes individuales de música rock.

- `album_id`: Identificador único del álbum (ej. A00001)  
- `album_title`: Título del álbum  
- `release_date`: Fecha de lanzamiento (mezcla de formatos como `dd/mm/yyyy` y `yyyy-mm-dd`)  
- `band_name`: Nombre de la banda que publicó el álbum  
- `genre`: Subgénero del rock (ej. Hard Rock, Progressive, Punk, etc.)  
- `sales`: Ventas totales del álbum (en formatos mixtos con comas o puntos)  
- `rating`: Valoración media del álbum (almacenada como texto)

---

#### `bands_info.csv`
Incluye información general sobre las bandas presentes en el dataset de álbumes.

- `band_id`: Identificador único de la banda (ej. B0001)  
- `band_name`: Nombre de la banda  
- `country`: País de origen  
- `active_years`: Rango de años activos (ej. "1970-2003")  
- `main_vocalist`: Nombre del vocalista principal

---

#### `reviews_rock.csv`
Contiene valoraciones de usuarios sobre diferentes álbumes.

- `review_id`: Identificador único de la reseña (ej. R0001)  
- `album_id`: ID del álbum valorado  
- `review_score`: Nota numérica entre 1.0 y 10.0  
- `would_recommend`: Si el usuario recomienda el álbum (`Y`, `N`, `yes`, `no`, con mayúsculas y minúsculas mezcladas)

---

Estos datasets han sido diseñados para simular datos reales, incluyendo inconsistencias y formatos diversos. Se espera que limpies, unas y analices estos datos usando las herramientas aprendidas durante el curso.

---

## Preguntas:

### i. Carga los datos 

Abre los tres datasets y muestra una vista previa de las **últimas 11 filas** de cada dataframe.  
Asegúrate de que los datos se hayan cargado correctamente.

In [ ]:
import pandas as pd

df_albums = pd.read_csv("albums_rock.csv")
df_bands = pd.read_csv("bands_info.csv")
df_reviews = pd.read_csv("reviews_rock.csv")

df_albums.tail(11), df_bands.tail(11), df_reviews.tail(11)

### 1. Limpia los datos 

Prepara los datos para su análisis. Atiende a los siguientes problemas:

**1.1. Algunos álbumes parecen aparecer más de una vez. Elimina duplicados.**

In [ ]:
df_albums["album_id"].duplicated().sum()

In [ ]:
df_albums = df_albums.drop_duplicates()
df_albums

In [ ]:
df_albums["album_id"].duplicated().sum()

Tiene sentido eliminar los duplicados y no imputar valores ya que los álbumes son el campo clave del dataset y no puede estar doble.

**1.2. La columna `rating` contiene texto o valores nulos. Convierte a numérico y sustituye los valores inválidos utilizando imputación de datos.**

In [ ]:
df_albums.isnull().sum()

In [ ]:
df_albums["rating"].unique()

In [ ]:
df_albums['rating'] = pd.to_numeric(df_albums['rating'], errors='coerce')
df_albums

In [ ]:
df_albums["rating"].isna().value_counts()

In [ ]:
df_albums.dtypes

In [ ]:
df_albums["rating"].hist(bins=15)

Podemos sustituir por la media o la mediana el rating, aunque apreciamos que no forma una campana de Gauss los valores. Analiza otras posibles imputaciones de datos que puedan tener sentido.

In [ ]:
df_albums_media = df_albums.copy()
df_albums_media["rating"]=df_albums_media["rating"].fillna(df_albums_media["rating"].mean())
df_albums_media.isnull().sum()

In [ ]:
df_albums_media["rating"].hist(bins=15)

**1.3. Convierte la columna `sales` a valores numéricos adecuados.**

Utiliza regex para la conversión de datos.

In [ ]:
df_albums = df_albums.copy()
df_albums['sales_clean'] = (
    df_albums['sales']
    .str.replace(r'\D', '', regex=True)  # elimina cualquier carácter no numérico
    .astype(float)
)

In [ ]:
df_albums = df_albums.copy()
df_albums['sales_clean_no_regex'] = (
    df_albums['sales']
    .str.replace('.', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
df_albums

In [ ]:
df_albums.dtypes

In [ ]:
df_albums.isnull().sum()

**1.4. Elimina cualquier columna que no aporte información útil.**

In [ ]:
if 'extra_notes' in df_albums.columns:
    df_albums = df_albums.drop(columns=['extra_notes'])
df_albums

**1.5. En el archivo de reseñas, convierte `review_score` a numérico e imputa valores si tiene sentido.**

In [ ]:
df_reviews['review_score'] = pd.to_numeric(df_reviews['review_score'], errors='coerce')
df_reviews['review_score'].isna().value_counts()

Tiene sentido eliminar la información ya que hay casi un 50% de valores nulos.

In [ ]:
df_reviews = df_reviews.dropna(subset=['review_score'])
df_reviews

---

### 2. ¿Cuántos álbumes se publicaron por género?

Muestra el número de álbumes por género. Ordena el resultado de mayor a menor.

In [ ]:
df_albums['genre'].value_counts()

---

### 3. ¿Cuál es el género con más ventas totales de álbumes?

In [ ]:
df_albums.groupby('genre')['sales_clean'].sum().sort_values(ascending=False)

---

### 4. ¿Cuál es la valoración media por género y país de origen de la banda?

Solo considera álbumes cuya `rating` esté entre 1.0 y 10.0.

In [ ]:
df_albums_bands = pd.merge(df_albums, df_bands, on='band_name', how='left')
valid = df_albums_bands[(df_albums_bands['rating'] >= 1.0) & (df_albums_bands['rating'] <= 10.0)]
valid.groupby(['genre', 'country'])['rating'].mean().round(2)

---

### 5. ¿Qué banda ha vendido más álbumes en total?

Muestra el nombre de la banda, el total vendido y el porcentaje que representa sobre el total global de ventas.

In [ ]:
df_albums.groupby('band_name')['sales_clean'].sum().sort_values(ascending=False).head(1)

---

### 6. ¿Cuántas reseñas fueron claramente positivas?

Considera positiva una reseña si el usuario indicó que recomendaría el álbum. Procesa la columna si es necesario.

In [ ]:
df_reviews['positive'] = df_reviews['would_recommend'].str.lower().isin(['y', 'yes'])
df_reviews['positive'].sum()

---

### 7. ¿Hay algún mes en que los álbumes lanzados hayan tenido mejor valoración que otros?

Agrupa la media de ratings por mes de publicación. Usa nombres de mes en el resultado. Muestra el mejor.

In [ ]:
df_albums['release_date'] = pd.to_datetime(df_albums['release_date'], errors='coerce', dayfirst=True)
df_albums['month'] = df_albums['release_date'].dt.strftime('%B')
df_albums.groupby('month')['rating'].mean().sort_values(ascending=False).round(2)

---

### 8. Dibuja un gráfico de barras con las ventas totales por género.

Etiqueta ejes, añade título y asegúrate de que las etiquetas no se solapen.

In [ ]:
import matplotlib.pyplot as plt

genre_sales = df_albums.groupby('genre')['sales_clean'].sum().sort_values()
plt.figure(figsize=(8,5))
genre_sales.plot(kind='barh', color='mediumslateblue')
plt.title('Total Album Sales by Genre')
plt.xlabel('Sales')
plt.ylabel('Genre')
plt.tight_layout()
plt.grid(True)
plt.show()